# Padronizando o arquivo de moveset (agrupando os dados por pokémon)

In [ ]:
import json

# Carrega o JSONL original e agrupar por Pokémon
pokemon_data = {}

with open('/home/yan/Documents/GitHub/competitive-poke-ai123/documents/54_pokemon_moveset_regmb.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        pkmn = entry['pokemon']
        category = entry['category']
        
        if pkmn not in pokemon_data:
            pokemon_data[pkmn] = {}
        
        # Guarda todas as categorias (stats, moves, items, etc.) completas
        pokemon_data[pkmn][category] = entry['metadata']

# Salva em um novo arquivo JSONL (1 objeto por Pokémon)
with open('/home/yan/Documents/GitHub/competitive-poke-ai123/documents/54_pokemon_moveset_regmb.jsonl', 'w', encoding='utf-8') as f:
    for pkmn, categories in pokemon_data.items():
        obj = {
            'pokemon': pkmn,
            'data': categories
        }
        f.write(json.dumps(obj, ensure_ascii=False) + '\n')

# Agrupa todos os .md referente aos pokémon singurmente para criar um arquivo de descrição

In [ ]:
import json
import re
from pathlib import Path

# Caminhos dos diretórios (ajustados para arquivos .md)
PASTA_MD = Path(
    "/home/yan/Documents/GitHub/competitive-poke-ai/Codes/data _preprocessing_and _acquisition/documents2"
)
PASTA_DOCS = Path(
    "/home/yan/Documents/GitHub/competitive-poke-ai/Codes/data _preprocessing_and _acquisition/teste2"
)
PASTA_DOCS.mkdir(parents=True, exist_ok=True)
CAMINHO_DESCRICOES = PASTA_DOCS / "file_descriptions.json"


def extrair_metadados(nome_arquivo):
    # Extrai regulamento e nome do Pokémon a partir do nome do arquivo .md
    padrao = r"^\d+_(reg[a-z0-9]+)_([^_]+)"
    match = re.search(padrao, nome_arquivo, re.IGNORECASE)
    if match:
        return match.group(1).lower(), match.group(2).strip()
    return "unknown", "unknown"


def extrair_role(texto):
    # Extrai a função competitiva (Competitive Role) do texto do .md se existir
    match = re.search(r"Competitive Role:\s*\n*\s*(.+)", texto, re.IGNORECASE)
    return match.group(1).strip() if match else "Unknown"


def consolidar_estrategias_md():
    arquivos_md = list(PASTA_MD.glob("*.md"))
    dados_por_reg = {}

    print(f"Lendo e processando {len(arquivos_md)} arquivos .md...")

    for arq in arquivos_md:
        reg, pokemon = extrair_metadados(arq.name)

        # Ignora arquivos que não sejam o padrão de estratégia (ex: 45_pokemon_chaos_regmb.md)
        if reg == "unknown" or pokemon == "unknown":
            continue

        try:
            conteudo = arq.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            conteudo = arq.read_text(encoding="latin-1")

        role = extrair_role(conteudo)

        # Limpa linhas vazias consecutivas mantendo a estrutura
        linhas_limpas = [l.strip() for l in conteudo.splitlines() if l.strip()]
        texto_limpo = "\n".join(linhas_limpas)

        registro = {
            "pokemon": pokemon,
            "regulation": reg.upper(),
            "role": role,
            "source_file": arq.name,
            "text": (
                f"Competitive Strategy for {pokemon} ({reg.upper()}) - Role:"
                f" {role}\n\n{texto_limpo}"
            ),
        }

        if reg not in dados_por_reg:
            dados_por_reg[reg] = []
        dados_por_reg[reg].append(registro)

    # Salva os ficheiros JSONL consolidados por regulamento
    descricoes_novas = {}
    for reg, registros in dados_por_reg.items():
        nome_jsonl = f"pokemon_strategies_{reg}.jsonl"
        caminho_jsonl = PASTA_DOCS / nome_jsonl

        with open(caminho_jsonl, "w", encoding="utf-8") as f:
            for item in registros:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        print(
            f"Criado: {nome_jsonl} ({len(registros)} estratégias .md salvas para"
            f" {reg.upper()})"
        )

        # Entrada correspondente para o file_descriptions.json
        descricoes_novas[nome_jsonl] = (
            f"Comprehensive Smogon competitive strategy guides and meta"
            f" analysis for Pokémon in {reg.upper()}. Includes competitive"
            " roles, item choices, EV/SP spreads, move choices, checks,"
            " counters, and team synergies. Use for qualitative strategy"
            " questions, team-building advice, role analysis, and counter"
            " matchups."
        )

    # Atualiza o file_descriptions.json
    if CAMINHO_DESCRICOES.exists():
        descricoes_existentes = json.loads(
            CAMINHO_DESCRICOES.read_text(encoding="utf-8")
        )
    else:
        descricoes_existentes = {}

    descricoes_existentes.update(descricoes_novas)

    CAMINHO_DESCRICOES.write_text(
        json.dumps(descricoes_existentes, ensure_ascii=False, indent=4),
        encoding="utf-8",
    )
    print(
        "\nSucesso! 'file_descriptions.json' atualizado com os novos ficheiros"
        " consolidados."
    )


if __name__ == "__main__":
    consolidar_estrategias_md()

Lendo e processando 70 arquivos .md...
Criado: 90_pokemon_strategies_regma.jsonl (39 estratégias .md salvas para REGMA)
Criado: 90_pokemon_strategies_regmb.jsonl (31 estratégias .md salvas para REGMB)

Sucesso! 'file_descriptions.json' atualizado com os novos ficheiros consolidados.
